# Лекција 13 - Сећање агента


## Подешавање

Овај нотебук демонстрира како направити агент за резервацију путовања са **постојећом меморијом** користећи **Microsoft Agent Framework** (MAF).

Учићете како различите врсте агентске меморије — радна, краткорочна и дугорочна — утичу на то како агент задржава и користи информације током разговора.

**Предуслови:**
- Azure AI Foundry пројекат са распоређеним моделом за чет (нпр. `gpt-4o-mini`).
- Пријављени у Azure CLI — покрените `az login` у вашем терминалу.
- `AZURE_AI_PROJECT_ENDPOINT` — крајња тачка вашег Azure AI Foundry пројекта.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — име вашег распоређеног модела.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Типови агенцијске меморије

АИ агенти могу користити различите врсте меморије, свака са својом одређеном сврхом:

### Радна меморија
Сам разговор — поруке размењене током једне сесије. Агент може да се позове на претходне поруке у истом разговору како би одржао кохерентност. У MAF-у ово се креира преко **`agent.create_session()`**, који враћа `AgentSession`.

### Краткорочна меморија
Информације које трају током задатка или сесије, али нису трајно сачуване. На пример, агент може да сакупљa чињенице током више корака планирања у разговору и искористи их за креирање коначног распореда путовања.

### Дугорочна меморија
Преференције и чињенице које трају **преко више сесија**. Корисник који се враћа не би требао да понавља своја дијетална ограничења или стил путовања. Дугорочна меморија се обично чува у спољном складишту — бази података, фајлу или векторском индексу — и агенту се представља путем алата.


## Радна меморија са сесијама

Најједноставнији облик меморије је сесија разговора. Када проследите исту сесију (создану преко `agent.create_session()`) узастопним позивима `agent.run()`, агент види пуну историју тог разговора и може да се сети ранијих детаља.

Хајде да направимо туристичког агента и демонстрирамо радну меморију.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Агент је исправно запамтио буџет јер обе поруке деле исту сесију. Ово је **радна меморија** — постоји само током трајања сесије.

### Шта се дешава са новом нитком?

Ако креирамо **нову** сесију, агент нема сећање претходног разговора:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Шема дугорочне меморије

Да бисмо памтили корисничке преференције **између сесија**, потребна нам је упорна меморија која живи ван нити разговора. Ајент приступа овој меморији преко **алата** — функција које може позвати да сачувају и преузму информације.

Испод имплементирамо једноставну преференцијску меморију у памћењу (у продукцији бисте користили базу података или векторски индекс) и представљамо је као алате које ајент може користити.

### Архитектура
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Сценарио 1 — Корисник који први пут резервише путовање поводом годишњице

Сара посећује сајт први пут. Агент треба да сачува њене преференције помоћу алата и искористи их да препоручи хотеле.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Сценарио 2 — Сара се враћа недељама касније

Сара покреће **потпуно нову тему** (симулирајући нову сесију). Радна меморија је празна, али дугорочна продавница преференција и даље има њене податке. Ајент би требало да их преузме и користи за персонализацију препорука.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Сажетак

У овој лекцији научили сте три типа агенцијске меморије и како их имплементирати уз Microsoft Agent Framework:

| Тип меморије | MAF механизам | Век трајања |
|---|---|---|
| **Радна** | `agent.create_session()` | Један разговор |
| **Краткорочна** | Накупљени контекст унутар нити | Један задатак / сесија |
| **Дугорочна** | Спољно складиште приступа се преко `@tool` функција | Преко сесија |

### Главни поенти
1. **`agent.create_session()`** обезбеђује радну меморију — агент види целу историју разговора унутар сесије.
2. **Нове сесије губе контекст** — без дугорочне меморије агент не може да се сети претходних разговора.
3. **`@tool` функције** превазилазе ту препреку — оне омогућавају агенту да чува и преузима информације из трајног складишта.
4. **Персонализација се побољшава временом** — што се више преференција сачува, боље су агентове препоруке.

### Примене у стварном свету
- **Корисничка подршка**: памћење историје и преференција корисника
- **Лични асистенти**: одржавање контекста кроз дане или недеље
- **Здравство**: праћење информација и преференција пацијената
- **Е-трговина**: персонализована куповина заснована на историји

### Следећи кораци
- Замена речничке меморије базом података или векторским складиштем (нпр. Azure AI Search)
- Додавање истека меморије за информације временски осетљиве природе
- Изградња мулти-агентских система са заједничком меморијом
- Истраживање Cognee нотебоока за меморију подржану графом знања


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем AI услуге за превођење [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, молимо вас да имате на уму да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом матерњем језику треба сматрати званичним извором. За критичне информације препоручује се професионални превод од стране стручних људи. Нисмо одговорни за било каква неспоразума или погрешне тумачења настала употребом овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
